# Experiment: Overcoming Optimization Degradation with ResNets

**Objective:**
To mathematically verify that Residual Connections (He et al., 2015) solve the degradation problem in ultra-deep neural networks.

**Hypothesis:**
A 12-layer 'Plain' network will struggle to optimize its training loss despite having Batch Normalization. Wrapping the layers in identity skip connections (ResNet) will allow the optimizer to route gradients cleanly, resulting in significantly lower training error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from model import build_plain_network, build_resnet
from core.losses import CategoricalCrossEntropy, Softmax
from core.optimizers import Adam

In [ ]:
print("Generating complex non-linear dataset...")
np.random.seed(42)
X_train = np.random.randn(1000, 20)
y_train = np.sum(np.sin(X_train) + np.square(X_train), axis=1)
y_train = np.digitize(y_train, bins=np.percentile(y_train, [25, 50, 75]))

print(f"Training samples: {X_train.shape[0]}")

In [ ]:
epochs = 40
batch_size = 64
learning_rate = 0.005

# Both networks have 12 layers (1 initial + 5 blocks of 2 + 1 output)
model_plain = build_plain_network(20, 64, 5, 4)
model_resnet = build_resnet(20, 64, 5, 4)

loss_fn = CategoricalCrossEntropy()
opt_plain = Adam(model_plain.get_parameters(), lr=learning_rate)
opt_resnet = Adam(model_resnet.get_parameters(), lr=learning_rate)

loss_history = {'plain': [], 'resnet': []}

print("Starting simultaneous training of 12-Layer architectures...")
for epoch in range(epochs):
    model_plain.train()
    model_resnet.train()
    
    epoch_loss_p = 0
    epoch_loss_r = 0
    batches = 0
    
    for i in range(0, X_train.shape[0], batch_size):
        X_b, y_b = X_train[i:i+batch_size], y_train[i:i+batch_size]
        batches += 1
        
        # Plain Model
        logits_p, cache_p = model_plain.forward(X_b)
        loss_p = loss_fn.forward(Softmax.forward(logits_p), y_b)
        model_plain.backward(loss_fn.backward(Softmax.forward(logits_p), y_b), cache_p)
        opt_plain.step(model_plain.get_parameters(), model_plain.get_gradients())
        epoch_loss_p += loss_p
        
        # ResNet Model
        logits_r, cache_r = model_resnet.forward(X_b)
        loss_r = loss_fn.forward(Softmax.forward(logits_r), y_b)
        model_resnet.backward(loss_fn.backward(Softmax.forward(logits_r), y_b), cache_r)
        opt_resnet.step(model_resnet.get_parameters(), model_resnet.get_gradients())
        epoch_loss_r += loss_r
        
    loss_history['plain'].append(epoch_loss_p / batches)
    loss_history['resnet'].append(epoch_loss_r / batches)

print(f"Final Training Loss -> Plain: {loss_history['plain'][-1]:.4f} | ResNet: {loss_history['resnet'][-1]:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(loss_history['plain'], label='12-Layer Plain Network', color='red', linewidth=2)
plt.plot(loss_history['resnet'], label='12-Layer ResNet', color='blue', linewidth=2)
plt.title("Optimization Degradation Problem: Plain vs ResNet")
plt.xlabel("Epochs")
plt.ylabel("Categorical Cross Entropy Loss")
plt.legend()
plt.grid(True)
plt.show()

## Scientific Conclusion

* **Degradation Problem:** The 12-layer Plain network experienced severe optimization difficulty. Despite having Batch Normalization, the solver failed to discover an optimal mapping through the vast sequence of non-linear layers.
* **Residual Success:** By routing the gradient backward through the explicit addition node (skip connection), the 12-layer ResNet effortlessly optimized the training error to near zero. The skip connection mathematically ensures the gradient scalar `+1` is passed untouched to lower layers.